In [49]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [50]:
# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transformation for frames
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [51]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None, num_frames=10):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        
        frames = self._extract_frames(video_path)
        return frames, label

    def _extract_frames(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idxs = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

        for i in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transform:
                    frame = self.transform(frame)
                frames.append(frame)
            else:
                break

        cap.release()

        # Pad if fewer frames
        while len(frames) < self.num_frames:
            frames.append(torch.zeros_like(frames[0]))

        return torch.stack(frames)  # Shape: (num_frames, 3, 224, 224)

In [52]:
real_path = r"C:\Users\rimjh\Downloads\dataset\videos_real"
fake_path = r"C:\Users\rimjh\Downloads\dataset\videos_fake"

real_videos = [os.path.join(real_path, f) for f in os.listdir(real_path) if f.endswith(".mp4")]
fake_videos = [os.path.join(fake_path, f) for f in os.listdir(fake_path) if f.endswith(".mp4")]

all_videos = real_videos + fake_videos
labels = [0] * len(real_videos) + [1] * len(fake_videos)

In [53]:
X_train, X_test, y_train, y_test = train_test_split(all_videos, labels, test_size=0.2, random_state=42)

train_dataset = VideoDataset(X_train, y_train, transform)
test_dataset = VideoDataset(X_test, y_test, transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [54]:
class CNN_LSTM(nn.Module):
    def __init__(self, cnn_model, hidden_size=256, num_classes=2):
        super(CNN_LSTM, self).__init__()
        self.cnn = nn.Sequential(*list(cnn_model.children())[:-1])  # Remove final FC
        self.lstm = nn.LSTM(input_size=2048, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):  # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        c_out = []
        for t in range(T):
            frame = x[:, t, :, :, :]  # (B, C, H, W)
            out = self.cnn(frame)     # (B, 2048, 1, 1)
            out = out.view(B, -1)     # (B, 2048)
            c_out.append(out)
        c_out = torch.stack(c_out, dim=1)  # (B, T, 2048)
        lstm_out, _ = self.lstm(c_out)     # (B, T, H)
        final_out = lstm_out[:, -1, :]     # Last time step
        return self.fc(final_out)

In [55]:
resnet = models.resnet50(pretrained=True)
model = CNN_LSTM(resnet).to(device)

In [56]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss_sum:.4f} | Accuracy: {correct/total:.4f}")

Epoch 1/5 | Loss: 17.2122 | Accuracy: 0.4286
Epoch 2/5 | Loss: 15.8015 | Accuracy: 0.3452
Epoch 3/5 | Loss: 14.3702 | Accuracy: 0.5595
Epoch 4/5 | Loss: 12.7479 | Accuracy: 0.6905
Epoch 5/5 | Loss: 12.9433 | Accuracy: 0.6548


In [57]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nTest Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds))


Test Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.50      0.36      0.42        11
           1       0.50      0.64      0.56        11

    accuracy                           0.50        22
   macro avg       0.50      0.50      0.49        22
weighted avg       0.50      0.50      0.49        22

